In [3]:
# STEP 1: CHECKS ONLY (fixed MultiIndex issue)
# ----------------------------------------

import yfinance as yf
import pandas as pd
import numpy as np

# --- Your coins ---
coins = [
    'BTC-USD', 'ETH-USD', 'BNB-USD', 'XRP-USD', 'SOL-USD', 'TRX-USD',
    'DOGE-USD', 'STETH-USD', 'ADA-USD', 'BCH-USD', 'LINK-USD', 'WETH-USD',
    'LEO-USD', 'SUN-USD', 'BETH-USD', 'XLM-USD', 'LTC-USD', 'DOT-USD',
    'HBAR-USD', 'NEAR-USD', 'ICP-USD', 'DAI-USD', 'ETC-USD', 'ONDO-USD',
    'KAS-USD', 'GT-USD', 'FTN-USD', 'ATOM-USD', 'RUNE-USD', 'AVAX-USD'
]

start_date = '2024-07-01'
end_date   = '2025-07-01'

# --- Download ---
frames = []
for coin in coins:
    print(f"Downloading {coin} ...")
    try:
        df_coin = yf.download(
            coin, start=start_date, end=end_date,
            interval='1d', progress=False, group_by='column'
        )
    except Exception as e:
        print(f"⚠️ ERROR downloading {coin}: {e}")
        continue

    if df_coin is None or df_coin.empty:
        print(f"⚠️ WARNING: No data for {coin}")
        continue

    # --- Flatten MultiIndex columns ---
    df_coin.columns = [col[0] if isinstance(col, tuple) else col for col in df_coin.columns]

    # Reset index
    df_coin = df_coin.reset_index()
    df_coin['Coin'] = coin
    frames.append(df_coin)

if not frames:
    raise RuntimeError("No data downloaded for any coin. Please check tickers or network.")

df_raw = pd.concat(frames, axis=0, ignore_index=True)

# --- Keep only needed columns ---
needed_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Coin']
missing_needed = [c for c in needed_cols if c not in df_raw.columns]
if missing_needed:
    raise KeyError(f"Missing expected columns in downloaded data: {missing_needed}\n"
                   f"Available columns: {list(df_raw.columns)}")

df_keep = df_raw[needed_cols].copy()

# --- Ensure dtypes are correct ---
df_keep['Date'] = pd.to_datetime(df_keep['Date'], errors='coerce')

features = ['Open', 'High', 'Low', 'Close', 'Volume']
for col in features:
    df_keep[col] = pd.to_numeric(df_keep[col], errors='coerce')

# --- Basic info per coin ---
print("\n--- BASIC COUNTS PER COIN ---")
counts = df_keep.groupby('Coin')['Date'].count().sort_values(ascending=False)
print(counts)

print("\n--- DATE RANGE PER COIN ---")
date_ranges = df_keep.groupby('Coin')['Date'].agg(['min', 'max'])
print(date_ranges)

# --- Missing values report ---
print("\n--- MISSING VALUES CHECK (counts per coin) ---")
missing_report = (
    df_keep
    .set_index('Coin')[features]
    .isna()
    .groupby(level=0)
    .sum()
    .astype(int)
    .sort_index()
)
print(missing_report)

# --- Outlier counts using 1.5*IQR ---
print("\n--- OUTLIER COUNTS CHECK (1.5 * IQR rule) ---")
outlier_counts = {}
for coin, group in df_keep.groupby('Coin'):
    outlier_counts[coin] = {}
    for feat in features:
        s = group[feat].dropna()
        if s.empty:
            outlier_counts[coin][feat] = 0
            continue
        Q1 = s.quantile(0.25)
        Q3 = s.quantile(0.75)
        IQR = Q3 - Q1
        if pd.isna(IQR) or IQR == 0:
            outlier_counts[coin][feat] = 0
            continue
        low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_out = ((s < low) | (s > high)).sum()
        outlier_counts[coin][feat] = int(n_out)

outlier_report = pd.DataFrame(outlier_counts).T.reindex(sorted(outlier_counts.keys()))
print(outlier_report)

# --- Save reports ---
missing_report.to_csv("missing_values_report.csv")
outlier_report.to_csv("outlier_report.csv")
date_ranges.to_csv("date_ranges_per_coin.csv")
counts.to_csv("row_counts_per_coin.csv")

# print("\n✅ Saved:")
# print(" - missing_values_report.csv")
# print(" - outlier_report.csv")
# print(" - date_ranges_per_coin.csv")
# print(" - row_counts_per_coin.csv")
# print("\n📣 Share the printed tables (or the CSVs) and I’ll write Step 2: cleaning + preprocessing code.")


--- BASIC COUNTS PER COIN ---
Coin
ADA-USD      365
ATOM-USD     365
XLM-USD      365
WETH-USD     365
TRX-USD      365
SUN-USD      365
STETH-USD    365
SOL-USD      365
RUNE-USD     365
ONDO-USD     365
NEAR-USD     365
LTC-USD      365
LINK-USD     365
LEO-USD      365
KAS-USD      365
ICP-USD      365
HBAR-USD     365
GT-USD       365
FTN-USD      365
ETH-USD      365
ETC-USD      365
DOT-USD      365
DOGE-USD     365
DAI-USD      365
BTC-USD      365
BNB-USD      365
BETH-USD     365
BCH-USD      365
AVAX-USD     365
XRP-USD      365
Name: Date, dtype: int64

--- DATE RANGE PER COIN ---
                 min        max
Coin                           
ADA-USD   2024-07-01 2025-06-30
ATOM-USD  2024-07-01 2025-06-30
AVAX-USD  2024-07-01 2025-06-30
BCH-USD   2024-07-01 2025-06-30
BETH-USD  2024-07-01 2025-06-30
BNB-USD   2024-07-01 2025-06-30
BTC-USD   2024-07-01 2025-06-30
DAI-USD   2024-07-01 2025-06-30
DOGE-USD  2024-07-01 2025-06-30
DOT-USD   2024-07-01 2025-06-30
ETC-USD   2024-0

In [5]:
import pandas as pd
import numpy as np

# Use the data from your Step 1 result (df_keep from previous cell)
features = ['Open', 'High', 'Low', 'Close', 'Volume']

def cap_outliers_iqr(df, group_col='Coin', features=features):
    def cap_group(g):
        capped = g.copy()
        for f in features:
            s = capped[f].dropna()
            if s.empty:
                continue
            Q1 = s.quantile(0.25)
            Q3 = s.quantile(0.75)
            IQR = Q3 - Q1
            low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
            capped[f] = np.clip(capped[f], low, high)
        return capped
    return df.groupby(group_col, group_keys=False).apply(cap_group).reset_index(drop=True)

df_capped = cap_outliers_iqr(df_keep, group_col='Coin', features=features)

# --- Check outliers after capping ---
print("\n--- OUTLIER COUNTS AFTER CAPPING (1.5 * IQR rule) ---")
outlier_counts = {}
for coin, group in df_capped.groupby('Coin'):
    outlier_counts[coin] = {}
    for feat in features:
        s = group[feat].dropna()
        if s.empty:
            outlier_counts[coin][feat] = 0
            continue
        Q1 = s.quantile(0.25)
        Q3 = s.quantile(0.75)
        IQR = Q3 - Q1
        if pd.isna(IQR) or IQR == 0:
            outlier_counts[coin][feat] = 0
            continue
        low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_out = ((s < low) | (s > high)).sum()
        outlier_counts[coin][feat] = int(n_out)
outlier_report = pd.DataFrame(outlier_counts).T.reindex(sorted(outlier_counts.keys()))
print(outlier_report)

# # Optionally, save capped dataset for use in later steps:
# df_capped.to_csv("assets/crypto_enriched_ohlcv_cleaned_capped.csv", index=False)
# print("\n✅ Outlier capped dataset saved as assets/crypto_enriched_ohlcv_cleaned_capped.csv")



--- OUTLIER COUNTS AFTER CAPPING (1.5 * IQR rule) ---
           Open  High  Low  Close  Volume
ADA-USD       0     0    0      0       0
ATOM-USD      0     0    0      0       0
AVAX-USD      0     0    0      0       0
BCH-USD       0     0    0      0       0
BETH-USD      0     0    0      0       0
BNB-USD       0     0    0      0       0
BTC-USD       0     0    0      0       0
DAI-USD       0     0    0      0       0
DOGE-USD      0     0    0      0       0
DOT-USD       0     0    0      0       0
ETC-USD       0     0    0      0       0
ETH-USD       0     0    0      0       0
FTN-USD       0     0    0      0       0
GT-USD        0     0    0      0       0
HBAR-USD      0     0    0      0       0
ICP-USD       0     0    0      0       0
KAS-USD       0     0    0      0       0
LEO-USD       0     0    0      0       0
LINK-USD      0     0    0      0       0
LTC-USD       0     0    0      0       0
NEAR-USD      0     0    0      0       0
ONDO-USD      0     0

In [9]:
df_capped.to_csv("../../Dashboard/assets/crypto_enriched_ohlcv_cleaned_datasets.csv", index=False)
print("\n✅ Outlier capped dataset saved as assets/crypto_enriched_ohlcv_cleaned_capped.csv")



✅ Outlier capped dataset saved as assets/crypto_enriched_ohlcv_cleaned_capped.csv
